# Apriori Algorithm

There are three parts to the Apriori Algorithm - support, confidence and lift. Let's take a closer look at each of these using a scenario.

A toy store wants to perform a market basket analysis to identify toys most likely to be purchased together. They collect data on a number of wooden building blocks that are sold individually. They want to know which block shapes are frequently purchased together. First, they calculate the frequency an item is purchased. This is called the support.

**The support is the number of transactions containing item X, divided by the total number of transactions.** In the diagram below, we are calculating the support for the round, red block.

In [1]:
import pandas as pd
from IPython.display import display, Math

df = pd.DataFrame({
    "Red Circle":        [1, 1, 1, 1, 0, 0, 0, 0],
    "Blue Triangle":     [1, 1, 1, 0, 1, 1, 1, 0],
    "Navy Star":         [1, 1, 0, 0, 1, 1, 0, 0],
    "Light Blue Square": [1, 0, 0, 0, 1, 0, 0, 0],
    "Pink Pentagon":     [0, 0, 0, 1, 0, 0, 0, 1],
    "Dark Red Square":   [0, 0, 0, 0, 1, 1, 1, 1],
}, index=[f"T{i}" for i in range(1, 9)])
df.index.name = "Transaction"

display(df)

,Red Circle,Blue Triangle,Navy Star,Light Blue Square,Pink Pentagon,Dark Red Square
Transaction,,,,,,
T1,1,1,1,1,0,0
T2,1,1,1,0,0,0
T3,1,1,0,0,0,0
T4,1,0,0,0,1,0
T5,0,1,1,1,0,1
T6,0,1,1,0,0,1
T7,0,1,0,0,0,1
T8,0,0,0,0,1,1


In [2]:
# One palette for both notebook HTML and PDF; TikZ shapes live in labstyle.sty.
shapes = {
    "Red Circle":        ("●", "#DC143C", r"\labcircle"),
    "Blue Triangle":     ("▲", "#1673FF", r"\labtriangle"),
    "Navy Star":         ("★", "#08275D", r"\labstar"),
    "Light Blue Square": ("■", "#86D9FA", r"\labsquare"),
    "Pink Pentagon":     ("⬟", "#FFB6C1", r"\labpentagon"),
    "Dark Red Square":   ("■", "#650018", r"\labsquare"),
}

def render_shape(value, column, output_format):
    if value == 0:
        return ""
    symbol, colour, latex_command = shapes[column]
    if output_format == "latex":
        return rf"\textcolor[HTML]{{{colour.lstrip('#')}}}{{\Large {latex_command}}}"
    return f'<span style="color:{colour}; font-size:26px">{symbol}</span>'

styled_df = df.style.format({
    col: (lambda value, col=col: render_shape(value, col, "html"))
    for col in df.columns
})
latex_style = df.style.format({
    col: (lambda value, col=col: render_shape(value, col, "latex"))
    for col in df.columns
})

display({
    "text/html": styled_df.to_html(),
    "text/latex": latex_style.to_latex(column_format="lcccccc"),
}, raw=True)

,Red Circle,Blue Triangle,Navy Star,Light Blue Square,Pink Pentagon,Dark Red Square
Transaction,,,,,,
T1,●,▲,★,■,,
T2,●,▲,★,,,
T3,●,▲,,,,
T4,●,,,,⬟,
T5,,▲,★,■,,■
T6,,▲,★,,,■
T7,,▲,,,,■
T8,,,,,⬟,■


In [3]:
# print(latex_style.to_latex()) # For debugging the PDF generation via LaTeX

## Verify support
For association rule mining, the support of an item is:

$$
\operatorname{Support}(X)=\frac{\text{Transactions containing }X}{\text{Total transactions}}
$$

For the red circle:

$$
\operatorname{Support}(\textcolor{red}{\bullet})=\frac{4}{8}=0.5
$$

This binary representation is particularly useful for **Apriori and association rule mining**, where you want to identify frequently occurring combinations of items.

In [4]:
# We use .mean() because the DataFrame contains binary values (0 and 1), and the mean of binary values is equivalent to the proportion of 1s.
support_red_circle = df["Red Circle"].mean()
#print(support_red_circle)  # 0.5

display(Math(
    rf"\operatorname{{Support}}(\textcolor{{red}}{{\bullet}}) = {support_red_circle}"
))

<IPython.core.display.Math object>



$$
\operatorname{Confidence}(
\{\textcolor{red}{\bullet}\}
\rightarrow
\{\textcolor{blue}{\blacktriangle}\}
)
=
\frac{
\operatorname{Support}(\{\textcolor{red}{\bullet},\textcolor{blue}{\blacktriangle}\})
}{
\operatorname{Support}(\{\textcolor{red}{\bullet}\})
}
=
\frac{3/8}{4/8}
=
\frac{3}{4}
=
0.75
$$



In [5]:
df[["Red Circle", "Blue Triangle"]]

,Red Circle,Blue Triangle
Transaction,,
T1,1,1
T2,1,1
T3,1,1
T4,1,0
T5,0,1
T6,0,1
T7,0,1
T8,0,0


In [6]:
# axis=1 evaluates the intersection of the two columns row by row. With axis=0 evaluates the intersection of all rows column by column.
df[["Red Circle", "Blue Triangle"]].all(axis=1).rename("Red Circle AND Blue Triangle")

Transaction
T1     True
T2     True
T3     True
T4    False
T5    False
T6    False
T7    False
T8    False
Name: Red Circle AND Blue Triangle, dtype: bool

In [7]:
support_red_circle_blue_triangle = df[["Red Circle", "Blue Triangle"]].all(axis=1).mean()
display(Math(
    rf"\operatorname{{Support}}(\textcolor{{red}}{{\bullet}},\textcolor{{blue}}{{\blacktriangle}}) = {support_red_circle_blue_triangle}"
))

display(Math(
    rf"\operatorname{{Confidence}}(\textcolor{{red}}{{\bullet}} \rightarrow \textcolor{{blue}}{{\blacktriangle}}) = \frac{{\operatorname{{Support}}(\textcolor{{red}}{{\bullet}},\textcolor{{blue}}{{\blacktriangle}})}}{{\operatorname{{Support}}(\textcolor{{red}}{{\bullet}})}} = \frac{{{support_red_circle_blue_triangle}}}{{{support_red_circle}}} = {support_red_circle_blue_triangle/support_red_circle}"
))
display()

<IPython.core.display.Math object>

<IPython.core.display.Math object>

The final step in the Apriori Algorithm is calculating lift. Lift reveals the strength of the rules we found when we calculated the support and confidence. Lift is calculated by dividing the X1 to X2 Confidence by the X2 Support. In the below formula, the denominator has the independent support values of the round, red block and the triangular, blue block. This calculation will give us the probability of both blocks being purchased together. However, if the denominator is more than the numerator, this does not indicate an association between the two items purchased but rather a random connection.

(Edureka, 2019)


$$
\operatorname{Lift}(
\{\textcolor{red}{\bullet}\}
\rightarrow
\{\textcolor{blue}{\blacktriangle}\}
)
=
\frac{
\operatorname{Support}(\{\textcolor{red}{\bullet},\textcolor{blue}{\blacktriangle}\})
}{
\operatorname{Support}(\{\textcolor{red}{\bullet}\})
\times
\operatorname{Support}(\{\textcolor{blue}{\blacktriangle}\})
}
=
\frac{3/8}{(4/8)\times(6/8)}
=
1
$$

This calculation shows there is no real association between the two blocks, even though at first glance they seem connected:

- the round red block and the triangular blue block are both frequently purchased items
- but a lift of exactly 1 means buying one of them tells us nothing about whether a customer will buy the other, the two purchases are independent.

This is why we calculate lift rather than stopping at support and confidence: two independently popular items can look associated without actually being so. A lift above 1 indicates a genuine association, that is what the toy store would need to see before changing its layout or marketing, and it is what we will look for in next week's lab.

In [8]:
support_blue_triangle = df["Blue Triangle"].mean()

display(Math(
    rf"\operatorname{{Lift}}(\textcolor{{red}}{{\bullet}} \rightarrow \textcolor{{blue}}{{\blacktriangle}}) = "
    rf"\frac{{\operatorname{{Support}}(\textcolor{{red}}{{\bullet}},\textcolor{{blue}}{{\blacktriangle}})}}"
    rf"{{\operatorname{{Support}}(\textcolor{{red}}{{\bullet}}) \times \operatorname{{Support}}(\textcolor{{blue}}{{\blacktriangle}})}}"
    rf" = \frac{{{support_red_circle_blue_triangle}}}{{{support_red_circle} \times {support_blue_triangle}}}"
    rf" = {support_red_circle_blue_triangle / (support_red_circle * support_blue_triangle):.2f}"
))


<IPython.core.display.Math object>